# Colab runner

We use this to run the heavy scripts on a Colab T4. The dataset and the cache are kept on Google Drive, so nothing is lost when the session ends.

First time only: set the runtime to T4 GPU and upload the Kaggle `dataset/` folder to `MyDrive/soilgsd/dataset/`.

In [ ]:
# ---- settings ----
REPO_URL = "https://github.com/MaTsuOHaruI/Coarse2Fine-SG.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soilgsd"    # holds dataset/ and cache/
REPO_DIR = "/content/soilgsd"

In [ ]:
# Mount Google Drive and check the dataset is there.
import os
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/cache", exist_ok=True)
assert os.path.exists(f"{DRIVE_ROOT}/dataset/Training_labels_updated.csv"), (
    f"Upload the Kaggle dataset folder to {DRIVE_ROOT}/dataset first"
)

# The code reads these two variables instead of hard-coded paths.
os.environ["SOILGSD_DATA_DIR"] = f"{DRIVE_ROOT}/dataset"
os.environ["SOILGSD_CACHE_DIR"] = f"{DRIVE_ROOT}/cache"

In [ ]:
# Clone the repository, or update it if this session already has it.
import subprocess

assert "OWNER/REPO" not in REPO_URL, "Set REPO_URL in the settings cell"
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret set: fine for a public repository

if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO_DIR], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Install the package. torch is not pinned, so Colab's preinstalled CUDA build is kept.
%cd $REPO_DIR
!pip install -q -e . pytest

In [ ]:
# Check the GPU and that the evaluation loop gives the reference scores (102.0 and 89.9).
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
!python scripts/eval_baselines.py

## SAM checkpoint

Goes into `cache/checkpoints/` on Drive (same path as `sam.checkpoint` in `configs/default.yaml`), so it is only downloaded once.

In [ ]:
import os, urllib.request
ckpt = f"{DRIVE_ROOT}/cache/checkpoints/sam_vit_b_01ec64.pth"
if not os.path.exists(ckpt):
    os.makedirs(os.path.dirname(ckpt), exist_ok=True)
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", ckpt + ".part")
    os.replace(ckpt + ".part", ckpt)
print(ckpt, os.path.getsize(ckpt) // 2**20, "MB")

## Scale table

Quick sanity check that every photo gets a px/mm value. Any other script can be run the same way.

In [ ]:
!python scripts/scale_table.py --config configs/default.yaml

## 1. Cache masks and features (GPU)

This is the slow part. Try `--limit 2` first to see that SAM loads and one photo goes through, then run the full thing. If Colab disconnects, just run the cell again: photos already in the cache are skipped.

In [ ]:
!python scripts/make_cache.py --segmenter sam --device cuda --limit 2

In [ ]:
# full run, train + test photos
!python scripts/make_cache.py --segmenter sam --device cuda

## 2. LOSO score

Leave-one-soil-out on the 24 training soils, read from the cache. For reference, the classical segmenter gave 89.89 / 42.77 / 43.65 / 43.72 (mean_curve / pure_b / adaptive / hard). Add `--dino` here and in step 1 to use DINOv2 features instead of the colour thumbnail.

In [ ]:
!python scripts/run_loso.py --segmenter sam --device cuda

In [ ]:
# runs/ is inside the cloned repo, which is gone after the session.
# Copy it to Drive and show the latest LOSO result.
import shutil
from pathlib import Path
import pandas as pd

def save_runs():
    shutil.copytree(f"{REPO_DIR}/runs", f"{DRIVE_ROOT}/runs", dirs_exist_ok=True)
    print("copied runs/ to", f"{DRIVE_ROOT}/runs")

def latest_run(name):
    found = sorted(Path(f"{REPO_DIR}/runs").glob(f"*_{name}"))
    assert found, f"no run named *_{name} yet"
    return found[-1]

save_runs()
loso_dir = latest_run("loso_sam")
print(loso_dir.name)
display(pd.read_csv(loso_dir / "summary.csv").round(2))
display(pd.read_csv(loso_dir / "per_soil.csv").round(2))

## 3. Submission

Pick the model with the best LOSO score above (not the leaderboard). The script checks the CSV before writing it.

In [ ]:
MODEL = "adaptive"    # mean_curve, pure_b, adaptive or hard
!python scripts/make_submission.py --model {MODEL} --segmenter sam --device cuda

In [ ]:
save_runs()
sub = latest_run(f"submission_{MODEL}") / "submission.csv"
print(sub)
display(pd.read_csv(sub).head())